# 01b — Pembangunan Ulang Dataset (satu pipeline, dapat diulang dari data mentah)

### Kriteria penerimaan (ditetapkan sebelum eksekusi; ubah hanya dengan alasan tertulis)
| # | Kriteria | Ambang |
|---|---|---|
| C1 | Selisih tingkat judul ganda antar label, tiap split | ≤ 5 poin |
| C2 | KS-statistic panjang kata antar label | ≤ 0,10 |
| C3 | Selisih tingkat teks berakhir tanda baca antar label | ≤ 5 poin |
| C4 | Akurasi pohon keputusan memakai fitur panjang saja (test) | ≤ 0,60 |
| C5 | Tidak ada group (klaster near-duplicate) yang melintasi split | 0 pelanggaran |
| C6 | Selisih proporsi sub-topik antar label | ≤ 2 poin |
| INFO | Akurasi 8-kata-pertama (TF-IDF+LR), ketimpangan sumber, tumpang-tindih tahun | dilaporkan, tanpa ambang |

In [1]:
import os, re, sys, json, hashlib, platform, datetime
import numpy as np, pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import ks_2samp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score

sys.path.insert(0, os.getcwd())
import text_pipeline as tp                      # <- modul yang sama dipakai backend

SEED = 42
RAW_DIR = os.environ.get("RAW_DIR") or ("../data/raw" if os.path.exists("../data/raw") else "data/raw")
OUT_DIR = os.environ.get("OUT_DIR") or ("../data/processed_v2" if os.path.exists("../data") else "data/processed_v2")
os.makedirs(OUT_DIR, exist_ok=True)

CFG = dict(MIN_WORDS=10, MAX_WORDS=60, NEAR_DUP_COS=0.90, GROUP_COS=0.70,
           LEN_BINS=[0, 20, 30, 40, 50, 60, 10_000], N_FOLDS=20, TRAIN_FOLDS=14, VAL_FOLDS=2, DEVTEST_FOLDS=2, LOCKBOX_FOLDS=2, SEED=SEED)

# Sumber data LENGKAP (3 Hoaks + 3 Fakta):
SOURCES = [
    dict(name="turnbackhoax.id",   label=1, file="dataset_turnbackhoax_10_cleaned", title=["Title"], body=["Clean Narasi", "Narasi"], date=["Timestamp", "Date", "Tanggal", "date"]),
    dict(name="komdigi.go.id",     label=1, file="komdigi_hoaks",                   title=["title"], body=["body_text", "excerpt"],   date=["published_at"]),
    dict(name="hoax_archive_15k",  label=1, file="hoax_dataset_with_text",          title=["judul"], body=["teks", "deskripsi"],     date=["tanggal"]),
    dict(name="tempo.co",          label=0, file="dataset_tempo_6k_cleaned",        title=["Title"], body=["FullText", "text_new"],  date=["Timestamp", "Date", "Tanggal", "date"]),
    dict(name="cnnindonesia.com",  label=0, file="dataset_cnn_10k_cleaned",         title=["Title"], body=["FullText", "text_new"],  date=["Timestamp", "Date", "Tanggal", "date"]),
    dict(name="kompas.com",        label=0, file="dataset_kompas_4k_cleaned",       title=["Title"], body=["FullText", "text_new"],  date=["Timestamp", "Date", "Tanggal", "date"]),
]
EXTRA_SOURCES = []      # contoh: dict(name="antaranews.com", label=0, file="dataset_antara", title=["Title"], body=["FullText"], date=["Date"])
USE_TITLE_ONLY_SOURCES = False   # True = ikutkan sumber tanpa isi (mis. Komdigi); default False

LOG = []
def log(step, df, note=""):
    g = df.groupby(["source", "label"]).size()
    for (s, l), n in g.items():
        LOG.append(dict(step=step, source=s, label=int(l), n=int(n), note=note))
    print(f"[{step:<28}] total={len(df):>7,} | per label={df.label.value_counts().sort_index().to_dict()}  {note}")

def show_retention():
    d = pd.DataFrame(LOG).groupby(["step", "label"]).n.sum().unstack().fillna(0)
    d = d.loc[list(dict.fromkeys(r["step"] for r in LOG))]
    out = d.astype(int).join((d.div(d.iloc[0]) * 100).round(1).add_suffix("_%dari_mentah"))
    print("\nRETENSI PER LANGKAH (filter yang menghantam satu label jauh lebih keras harus dibenarkan tertulis di PROTOKOL):")
    print(out.to_string())

In [2]:
# ---------------- 1. MUAT DATA MENTAH (tanpa menimpa apa pun) ----------------
import re

# Regex ekstraksi klaim di dalam tanda petik dari narasi hoaks
QUOTE_EXTRACT = re.compile(
    r'(?:berisi\s+(?:informasi|narasi|klaim)|narasi|klaim)\s*[:\s]*["\'\u201c\u2018]([^"\'\u201d\u2019]{15,600})["\'\u201d\u2019]',
    re.I
)
TAG_CLEAN = re.compile(r"^\s*(?:[\[\(\{][^\]\)\}]{1,25}[\]\)\}]\s*:?\s*)+")

def extract_hoax_claim(judul: str, teks: str) -> str:
    """Ekstrak klaim asli di dalam kutipan teks; jika tidak ada, fallback ke judul bersih."""
    judul_clean = TAG_CLEAN.sub("", str(judul)).strip()
    m = QUOTE_EXTRACT.search(str(teks))
    if m:
        extracted = m.group(1).strip()
        if len(extracted.split()) >= 6:
            return extracted
    return ""

def pick(row, names):
    for n in names:
        if n in row.index:
            v = row[n]
            if pd.notna(v) and str(v).strip() not in ("", "nan", "None"):
                return str(v).strip()
    return ""

def load_source(s):
    files = sorted(f for f in os.listdir(RAW_DIR) if f.startswith(s["file"] + ".")) or \
            sorted(f for f in os.listdir(RAW_DIR) if f.startswith(s["file"]))
    if not files:
        print(f"!! BERKAS TIDAK DITEMUKAN untuk {s['name']}: {s['file']}"); return pd.DataFrame()
    p = os.path.join(RAW_DIR, files[0])
    d = pd.read_excel(p) if p.endswith((".xlsx", ".xls")) else pd.read_csv(p)
    
    titles = [pick(r, s["title"]) for _, r in d.iterrows()]
    bodies = [pick(r, s["body"])  for _, r in d.iterrows()]
    date_raws = [pick(r, s["date"]) for _, r in d.iterrows()]
    
    # KHUSUS hoax_archive_15k: Ambil kutipan klaim asli atau fallback judul (bukan teks bantahan jurnalis)
    if s["name"] == "hoax_archive_15k":
        bodies = [extract_hoax_claim(t, b) for t, b in zip(titles, bodies)]
    
    out = pd.DataFrame({
        "title":    titles,
        "body":     bodies,
        "date_raw": date_raws,
    })
    
    _parsed = pd.to_datetime(out["date_raw"], errors="coerce", format="mixed")
    _years = out["date_raw"].astype(str).str.extract(r'\b(20\d\d)\b')[0]
    out["date"] = _parsed.fillna(pd.to_datetime(_years + "-01-01", errors="coerce"))
    out["source"], out["label"], out["file"] = s["name"], s["label"], files[0]
    print(f"{s['name']:<18} {files[0]:<45} baris={len(out):>6,} | isi kosong={ (out.body=='').mean()*100:5.1f}% | tanggal terbaca={out.date.notna().mean()*100:5.1f}%")
    return out

raw = pd.concat([load_source(s) for s in SOURCES + EXTRA_SOURCES], ignore_index=True)
log("1_raw", raw)

no_body = raw.assign(e=(raw.body == "")).groupby("source").e.mean()
title_only = no_body[no_body > 0.95].index.tolist()
if title_only and not USE_TITLE_ONLY_SOURCES:
    print(f"\n>> Sumber tanpa isi dikeluarkan (USE_TITLE_ONLY_SOURCES=False): {title_only}")
    raw = raw[~raw.source.isin(title_only)].reset_index(drop=True)
    log("1b_tanpa_sumber_judul_saja", raw)

turnbackhoax.id    dataset_turnbackhoax_10_cleaned.csv           baris=10,381 | isi kosong= 37.3% | tanggal terbaca=100.0%
komdigi.go.id      komdigi_hoaks.csv.csv                         baris= 3,624 | isi kosong=  0.0% | tanggal terbaca=100.0%
hoax_archive_15k   hoax_dataset_with_text (2).csv.csv            baris=15,720 | isi kosong= 68.3% | tanggal terbaca=100.0%
tempo.co           dataset_tempo_6k_cleaned.csv                  baris= 6,592 | isi kosong=  0.0% | tanggal terbaca=100.0%
cnnindonesia.com   dataset_cnn_10k_cleaned.csv                   baris= 9,630 | isi kosong=  0.0% | tanggal terbaca=100.0%
kompas.com         dataset_kompas_4k_cleaned.csv                 baris= 4,750 | isi kosong=  0.6% | tanggal terbaca= 99.6%
[1_raw                       ] total= 50,697 | per label={0: 20972, 1: 29725}  


In [3]:
# ---------------- 2. BENTUK TEKS (aturan sama untuk semua kelas) -------------
raw["title_c"] = raw.title.map(tp.clean)
raw["body_c"]  = [tp.strip_leading_title(t, tp.clean(b)) for t, b in zip(raw.title_c, raw.body)]
raw["text"] = [tp.build_text(t, b) for t, b in zip(raw.title, raw.body)]
print("\nTag kategori di awal judul/isi (dibuang untuk SEMUA kelas oleh tp.clean), per label dan sumber:")
print(raw.assign(tag_judul=raw.title.map(tp.has_leading_tag), tag_isi=raw.body.map(tp.has_leading_tag))
         .groupby(["label", "source"])[["tag_judul", "tag_isi"]].sum().to_string())
raw["n_words_raw"] = raw.text.str.split().str.len()

repeat = pd.Series([bool(tp._alnum(b)) and tp._alnum(b).startswith(tp._alnum(tp.clean(t))) and bool(tp._alnum(t))
                    for t, b in zip(raw.title, raw.body.map(tp.clean))], index=raw.index)
print("\nBody diawali judul (judul diulang di sumber), per sumber:")
print(raw.assign(rep=repeat).groupby("source")["rep"].mean().mul(100).round(1).to_string())

df = raw[raw.text.str.len() > 0].copy()
log("2_teks_terbentuk", df)
df = df[df.n_words_raw >= CFG["MIN_WORDS"]].copy()
log("2b_min_kata", df, f">= {CFG['MIN_WORDS']} kata")


Tag kategori di awal judul/isi (dibuang untuk SEMUA kelas oleh tp.clean), per label dan sumber:
                        tag_judul  tag_isi
label source                              
0     cnnindonesia.com          0        0
      kompas.com               58        0
      tempo.co                  0        0
1     hoax_archive_15k      15599        7
      komdigi.go.id          3622        0
      turnbackhoax.id       10264       45

Body diawali judul (judul diulang di sumber), per sumber:
source
cnnindonesia.com    0.0
hoax_archive_15k    3.7
komdigi.go.id       0.0
kompas.com          0.0
tempo.co            0.0
turnbackhoax.id     4.1
[2_teks_terbentuk            ] total= 50,674 | per label={0: 20951, 1: 29723}  
[2b_min_kata                 ] total= 38,487 | per label={0: 20947, 1: 17540}  >= 10 kata


# AUDIT PEMOTONGAN DEBUNK  — sisipkan di 01b TEPAT SETELAH sel "2. BENTUK TEKS"

In [4]:
# =============================================================================
# SEL AUDIT PEMOTONGAN DEBUNK  — sisipkan di 01b TEPAT SETELAH sel "2. BENTUK TEKS"
# Hanya membaca. Memakai `raw`, `tp`, `CFG` yang sudah ada. Tidak mengubah data.
# Menjawab: seberapa besar pemotongan "bantahan" menghantam tiap label/sumber, kata apa pemicunya,
# apa yang dibuang dari FAKTA (positif palsu), dan seberapa besar kebocoran kata-vonis yang MASIH tersisa.
# =============================================================================
import re
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

pat_cut = getattr(tp, "DEBUNK_CUT", None)
pat_head = getattr(tp, "DEBUNKER_HEADER", None)
print("DEBUNK_CUT     :", pat_cut.pattern if pat_cut else None)
print("DEBUNKER_HEADER:", pat_head.pattern if pat_head else None)
assert pat_cut is not None, "tp.DEBUNK_CUT tidak ditemukan"

W = lambda s: len(str(s).split())
a = raw[["title", "body", "label", "source"]].copy()
a["body"] = a.body.astype(str); a["tw"] = a.title.map(W); a["w0"] = a.body.map(W)

def apply_all(s):                                   # urutan sama dengan pipeline: header lalu potong
    if pat_head is not None: s = pat_head.sub("", s)
    return pat_cut.sub("", s)

def trigger(s):
    m = pat_cut.search(s)
    return " ".join(m.group(0).lower().split()[:2]) if m else None

a["trig"] = a.body.map(trigger); a["hit"] = a.trig.notna()
a["w1"] = a.body.map(lambda s: W(apply_all(s)))
MINW = CFG["MIN_WORDS"]
a["dropped_by_cut"] = ((a.tw + a.w1) < MINW) & ((a.tw + a.w0) >= MINW)       # baris yang HILANG hanya karena pemotongan

# ---- 1. Dampak per label x sumber ----
t1 = a.groupby(["label", "source"]).agg(n=("hit", "size"), terpotong_pct=("hit", lambda x: round(x.mean() * 100, 1)),
        median_kata_sebelum=("w0", "median"), median_kata_sesudah=("w1", "median"),
        hilang_karena_potong_pct=("dropped_by_cut", lambda x: round(x.mean() * 100, 1)))
print("\n[1] DAMPAK PEMOTONGAN per label x sumber (filter yang menghantam satu label jauh lebih keras = asimetri):"); print(t1.to_string())
print("\nRingkas per label:"); print(a.groupby("label").agg(terpotong_pct=("hit", lambda x: round(x.mean() * 100, 1)),
        hilang_karena_potong_pct=("dropped_by_cut", lambda x: round(x.mean() * 100, 1))).to_string())

# ---- 2. Kata pemicu per label ----
t2 = a[a.hit].groupby(["trig", "label"]).size().unstack(fill_value=0)
t2 = t2.reindex(columns=[0, 1], fill_value=0).assign(total=lambda x: x.sum(axis=1)).sort_values("total", ascending=False).head(15)
print("\n[2] Pemicu terbanyak per label (kolom 0 = FAKTA, 1 = HOAKS). Pemicu yang banyak di kolom 0 = positif palsu:"); print(t2.to_string())

# ---- 3. Apa yang DIBUANG dari FAKTA (baca dengan mata) ----
print("\n[3] Contoh bagian yang dibuang dari FAKTA:")
for _, r in a[(a.label == 0) & a.hit].sample(min(8, int(((a.label == 0) & a.hit).sum())), random_state=1).iterrows():
    m = pat_cut.search(r.body)
    print(f"  [{r.source}] sisa: ...{r.body[:m.start()][-70:]!r} || DIBUANG({W(r.body[m.start():])} kata): {r.body[m.start():m.start()+90]!r}")
print("\n    Contoh yang dibuang dari HOAKS:")
for _, r in a[(a.label == 1) & a.hit].sample(min(5, int(((a.label == 1) & a.hit).sum())), random_state=1).iterrows():
    m = pat_cut.search(r.body)
    print(f"  [{r.source}] sisa: ...{r.body[:m.start()][-70:]!r} || DIBUANG({W(r.body[m.start():])} kata): {r.body[m.start():m.start()+90]!r}")

# ---- 4. Sensitivitas: kebocoran kata-vonis yang MASIH tersisa (leksikon tetap, ditetapkan sebelum melihat hasil) ----
VERDICT = ["hoaks", "hoax", "palsu", "keliru", "tidak benar", "menyesatkan", "disinformasi", "misinformasi", "klarifikasi", "bohong",
           "faktanya", "periksa fakta", "cek fakta", "penjelasan", "fakta sebenarnya", "tidak terbukti", "tidak ditemukan"]
rx = re.compile(r"(?<!\w)(?:" + "|".join(re.escape(v) for v in VERDICT) + r")(?!\w)", re.I)
variants = {"A. tanpa pemotongan (isi mentah)": a.body,
            "B. setelah header+pemotongan (pipeline)": a.body.map(apply_all),
            "C. B + kata-vonis ditutup (diganti 'x')": a.body.map(apply_all).map(lambda s: rx.sub("x", s))}
samp = pd.concat([g.sample(min(len(g), 5000), random_state=0) for _, g in a.groupby("label")])
rows = []
for nm, ser in variants.items():
    txt = samp.index.map(lambda i: ser.loc[i]); y = samp.label.to_numpy()
    xtr, xte, ytr, yte = train_test_split(list(txt), y, test_size=0.3, stratify=y, random_state=0)
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    acc = LogisticRegression(max_iter=2000).fit(v.fit_transform(xtr), ytr).score(v.transform(xte), yte)
    leak = pd.Series(list(txt)).map(lambda s: bool(rx.search(s))).groupby(y).mean() * 100
    rows.append((nm, round(acc * 100, 2), round(leak.get(0, 0), 1), round(leak.get(1, 0), 1)))
print("\n[4] Probe linier pada ISI (split acak, hanya untuk membandingkan varian; BUKAN estimasi performa):")
print(pd.DataFrame(rows, columns=["varian", "akurasi_%", "kata_vonis_fakta_%", "kata_vonis_hoaks_%"]).to_string(index=False))
print("    Baca: bila C jauh di bawah B, kebocoran vonis masih menggerakkan klasifikator. Bila C tetap tinggi, sinyal sumber/gaya yang tersisa bukan kata-vonis.")
print("    Catatan: B bisa 'bersih' dari 'faktanya' secara TAUTOLOGIS (pemotongan menghapus semua teks sesudahnya). Periksa kolom hilang_karena_potong_pct, bukan hanya frekuensi kata.")

DEBUNK_CUT     : \n?\s*(?:namun\s+|tetapi\s+)?(PENJELASAN|FAKTANYA|HASIL PERIKSA FAKTA|KLARIFIKASI|HOAKS ATAU FAKTA\?|KESIMPULAN|ANALISIS)\s*:.*
DEBUNKER_HEADER: ^\s*hasil\s+(?:periksa|cek)\s+fakta\s+[^\.\n]+(?:\([^\)]+\))?[\.\:\,\-]?\s*(?:(?:informasi|konten|klaim|kabar|video|narasi)\s+tersebut\s+(?:hoaks|hoax|palsu|tidak\s+benar|keliru|disinformasi)[\.\:\,\-]?)?\s*

[1] DAMPAK PEMOTONGAN per label x sumber (filter yang menghantam satu label jauh lebih keras = asimetri):
                            n  terpotong_pct  median_kata_sebelum  median_kata_sesudah  hilang_karena_potong_pct
label source                                                                                                    
0     cnnindonesia.com   9630            0.0                310.0                310.0                       0.0
      kompas.com         4750            0.0                250.0                250.0                       0.0
      tempo.co           6592            0.1                353.0      

# Uji Teks

In [5]:
# Sel tambahan untuk menjawab permintaan reviewer:
import os, pandas as pd

raw_dir = "data/raw" if os.path.exists("data/raw") else "../data/raw"
f_hoax = [f for f in os.listdir(raw_dir) if f.startswith("hoax_dataset_with_text")][0]
df_h = pd.read_csv(os.path.join(raw_dir, f_hoax), nrows=5)

print("=" * 80)
print("5 BARIS MENTAH hoax_archive_15k (judul, deskripsi, teks):")
print("=" * 80)
for i, r in df_h.iterrows():
    print(f"\n--- Sampel #{i+1} ---")
    print("  [judul]    :", str(r.get("judul", ""))[:200])
    print("  [deskripsi]:", str(r.get("deskripsi", ""))[:200])
    print("  [teks]     :", str(r.get("teks", ""))[:200])

5 BARIS MENTAH hoax_archive_15k (judul, deskripsi, teks):

--- Sampel #1 ---
  [judul]    : [SALAH] Salam 3 Jari dari Anies Bentuk Dukungan ke Pramono—Rano
  [deskripsi]: Faktanya, belum ada dukungan resmi Anies untuk pasangan manapun di Pilkada Jakarta 2024.
  [teks]     : Faktanya, belum ada dukungan resmi Anies untuk pasangan manapun di Pilkada Jakarta 2024.Unggahan berisi informasi “Anies Baswedan mendukung calon gubernur dan wakil gubernur nomor urut 3 Pramono—Rano”

--- Sampel #2 ---
  [judul]    : [SALAH] Prabowo Ancam Penghujat Pemimpin akan Berurusan dengan Aparatur Negara
  [deskripsi]: Tidak ada sumber valid yang membenarkan Presiden Prabowo menyampaikan ancaman tersebut.
  [teks]     : Tidak ada sumber valid yang membenarkan Presiden Prabowo menyampaikan ancaman tersebut.Narasi “Presiden Prabowo sebut rakyat akan berurusan dengan aparatur negara jika menghujat pemimpin” merupakan ko

--- Sampel #3 ---
  [judul]    : [SALAH] Menkes Budi Sebut Lockdown Pandemi Selanjutnya Bag

In [6]:
# ---------------- 3. SUB-TOPIK: word-boundary (bukan substring) --------------
SUB_TOPIC_RULES = {
    "pemilu": ["pemilu", "pilpres", "pilkada", "kpu", "bawaslu", "surat suara", "sirekap", "coblos", "capres", "cawapres", "cawagub", "caleg", "debat", "partai", "pdip", "gerindra", "golkar", "nasdem", "pks", "demokrat", "kpud", "panwaslu", "koalisi", "oposisi", "cawe-cawe", "dinasti", "legislatif"],
    "kebijakan": ["bansos", "subsidi", "bbm", "ikn", "apbn", "apbd", "pajak", "omnibus", "undang-undang", "perpres", "perda", "tarif listrik", "bpjs", "kebijakan pemerintah", "kementerian", "pemerintah", "negara", "asn", "bumn", "pertamina", "bulog", "beras bansos", "makan siang", "kemensos", "kemenkeu", "kemendagri", "reshuffle", "kabinet", "istana negara", "anggaran"],
    "tokoh": ["jokowi", "prabowo", "gibran", "anies", "ganjar", "megawati", "menteri", "presiden", "pejabat", "mahfud", "gubernur", "bupati", "walikota", "dpr", "mpr", "dprd", "puan", "erick thohir", "bahlil", "kaesang", "ridwan kamil", "ahok", "khofifah", "kades", "sri mulyani", "luhut", "nadiem", "basuki", "tito karnavian", "yasonna", "paspampres"],
    "hukum": ["korupsi", "kpk", "mk", "mahkamah", "polri", "kejaksaan", "tangkap", "ott", "sidang", "vonis", "hakim", "polisi", "tni", "aparat", "kejagung", "jaksa", "kasasi", "mahkamah agung", "yudikatif"],
}
NOISE = ["hujan es", "imunitas", "kanker", "diabetes", "ramuan", "vaksin hpv", "stroke", "gempa bumi", "tsunami", "cuaca ekstrem",
         "pelatih timnas", "liga 1", "piala aff", "frank de boer", "ronaldo", "messi", "shin tae-yong", "pesulap", "zodiak",
         "air kelapa", "janin", "kehamilan", "rahim", "kolesterol", "asam urat","lowongan", "loker", "rekrutmen", "pendaftaran kerja", 
         "pt pertamina persero loker", "seleksi bumn"]
wb = lambda words: re.compile(r"(?<!\w)(?:" + "|".join(re.escape(w) for w in words) + r")(?!\w)", re.I)
RULES_WB = {k: wb(v) for k, v in SUB_TOPIC_RULES.items()}; NOISE_WB = wb(NOISE)

def topic_wb(t):
    if NOISE_WB.search(t): return "bukan_politik"
    for k, rx in RULES_WB.items():
        if rx.search(t): return k
    return "umum"
def topic_substring(t):                                   # perilaku LAMA (hanya untuk perbandingan)
    t = t.lower()
    if any(w in t for w in NOISE): return "bukan_politik"
    for k, ws in SUB_TOPIC_RULES.items():
        if any(w in t for w in ws): return k
    return "umum"

df["sub_topic"] = df.text.map(topic_wb)
old = df.text.map(topic_substring)
changed = (old != df.sub_topic)
print(f"\nTopik berubah bila substring -> word-boundary: {changed.sum():,} teks ({changed.mean()*100:.1f}%). Contoh pemicu substring:")
print(df[changed].assign(lama=old[changed]).groupby(["lama", "sub_topic"]).size().sort_values(ascending=False).head(6).to_string())

df = df[~df.sub_topic.isin(["umum", "bukan_politik"])].copy()
log("3_filter_politik", df)


Topik berubah bila substring -> word-boundary: 2,499 teks (6.5%). Contoh pemicu substring:
lama           sub_topic
hukum          umum         632
kebijakan      umum         475
               tokoh        371
tokoh          umum         311
bukan_politik  pemilu       241
pemilu         kebijakan    195
[3_filter_politik            ] total= 27,187 | per label={0: 20349, 1: 6838}  


In [7]:
# ---------------- 4. BATAS PANJANG SAMA + DEDUP + GRUP ----------------------
df["text"] = df.text.map(lambda t: tp.cap_sentences(t, CFG["MAX_WORDS"]))
df["n_words"] = df.text.str.split().str.len()
df = df[df.n_words >= CFG["MIN_WORDS"]].reset_index(drop=True)
log("4_batas_panjang_sama", df, f"maks {CFG['MAX_WORDS']} kata, semua label")

before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
log("4b_duplikat_persis", df, f"dibuang {before-len(df):,}")

def similarity_edges(texts, thr, chunk=500):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.5, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform(texts).tocsr(); XT = X.T.tocsr()
    I, J, V = [np.empty(0, int)], [np.empty(0, int)], [np.empty(0, np.float32)]
    for s in range(0, X.shape[0], chunk):
        S = (X[s:s + chunk] @ XT).tocoo()
        m = (S.data >= thr) & (S.row + s < S.col)
        I.append(S.row[m] + s); J.append(S.col[m]); V.append(S.data[m])
    return np.concatenate(I), np.concatenate(J), np.concatenate(V)

def comp(n, I, J):
    return connected_components(coo_matrix((np.ones(len(I)), (I, J)), shape=(n, n)), directed=False)[1]

n = len(df); lab = df.label.to_numpy()
I, J, V = similarity_edges(df.text.tolist(), CFG["GROUP_COS"])
print(f"\nPasangan mirip (cos >= {CFG['GROUP_COS']}): {len(I):,}")

hi = V >= CFG["NEAR_DUP_COS"]
keep = np.ones(n, bool)
cross = hi & (lab[I] != lab[J])                          # mirip tapi beda label -> ambigu: buang keduanya
keep[I[cross]] = False; keep[J[cross]] = False
same = hi & (lab[I] == lab[J])
if same.any():
    c = comp(n, I[same], J[same]); first = pd.Series(np.arange(n)).groupby(c).transform("min").to_numpy()
    touched = np.zeros(n, bool); touched[I[same]] = True; touched[J[same]] = True
    keep &= ~(touched & (np.arange(n) != first))
print(f"Near-duplicate cos>={CFG['NEAR_DUP_COS']}: beda-label dibuang={int(cross.sum()):,} pasangan | sama-label dibuang={int((~keep).sum()):,} baris total")

old_idx = np.flatnonzero(keep); remap = -np.ones(n, int); remap[old_idx] = np.arange(len(old_idx))
m = keep[I] & keep[J]
df = df.loc[keep].reset_index(drop=True)
g = comp(len(df), remap[I[m]], remap[J[m]])
df["group_id"] = g
sizes = pd.Series(g).value_counts()
print(f"Grup near-duplicate: {len(sizes):,} grup | terbesar={sizes.iloc[0]} | grup berukuran >1: {(sizes>1).sum():,}")
log("4c_dedup_near", df)

[4_batas_panjang_sama        ] total= 27,187 | per label={0: 20349, 1: 6838}  maks 60 kata, semua label
[4b_duplikat_persis          ] total= 26,464 | per label={0: 20349, 1: 6115}  dibuang 723

Pasangan mirip (cos >= 0.7): 1,500
Near-duplicate cos>=0.9: beda-label dibuang=0 pasangan | sama-label dibuang=692 baris total
Grup near-duplicate: 25,076 grup | terbesar=8 | grup berukuran >1: 646
[4c_dedup_near               ] total= 25,772 | per label={0: 20318, 1: 5454}  


In [8]:
# ---------------- 5. AUDIT SEBELUM PENYEIMBANGAN (tidak ada filter di sini) ---
def audit(d, title):
    print("\n" + "=" * 78); print(title); print("=" * 78)
    a = d.assign(dup=d.text.map(tp.dup_title), endp=d.text.str.endswith((".", "!", "?")),
                 w=d.text.str.split().str.len()).groupby("label")
    print(pd.DataFrame({"n": a.size(), "judul_ganda_%": a.dup.mean()*100, "berakhir_tanda_%": a.endp.mean()*100,
                        "kata_mean": a.w.mean(), "kata_sd": a.w.std(), "kata_median": a.w.median(), "kata_max": a.w.max()}).round(2).to_string())
    print("\nsub_topik x label:"); print(pd.crosstab(d.sub_topic, d.label).to_string())
    print("\nsumber x label:");   print(pd.crosstab(d.source, d.label).to_string())
    if d.date.notna().any():
        print("\ntahun x label (dari kolom tanggal, hanya yang terbaca):")
        print(pd.crosstab(d.date.dt.year, d.label).to_string())
    print("\nKata pembuka teratas per label:")
    for l in (0, 1):
        print(f"  label {l}:", d[d.label == l].text.str.split().str[0].str.lower().value_counts().head(8).to_dict())

audit(df, "AUDIT SEBELUM PENYEIMBANGAN")


AUDIT SEBELUM PENYEIMBANGAN
           n  judul_ganda_%  berakhir_tanda_%  kata_mean  kata_sd  kata_median  kata_max
label                                                                                   
0      20318           2.22             100.0      50.84     6.80         52.0        60
1       5454          10.76             100.0      27.59    16.91         22.0        60

sub_topik x label:
label          0     1
sub_topic             
hukum         98   611
kebijakan   3364  1426
pemilu     16209   922
tokoh        647  2495

sumber x label:
label                0     1
source                      
cnnindonesia.com  9448     0
hoax_archive_15k     0  2021
komdigi.go.id        0   443
kompas.com        4449     0
tempo.co          6421     0
turnbackhoax.id      0  2990

tahun x label (dari kolom tanggal, hanya yang terbaca):
label      0    1
date             
2015       0   25
2016       0  157
2017     327  289
2018     741  496
2019     694  588
2020     500  908
2021   

In [9]:
# ---------------- 6. PENYEIMBANGAN TERKENDALI + SPLIT BERKELOMPOK -----------
df["len_bin"] = pd.cut(df.n_words, bins=CFG["LEN_BINS"]).astype(str)

def coarsened_match(d, keys, seed):
    parts = []
    for _, gk in d.groupby(keys, observed=True):
        c = gk.label.value_counts()
        if len(c) < 2: continue
        k = int(c.min())
        parts += [gg.sample(k, random_state=seed) for _, gg in gk.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

bal = coarsened_match(df, ["sub_topic", "len_bin"], SEED)
log("6_seimbang_topik_panjang", bal, "strata = sub_topic x len_bin")
print(f"Ukuran: {len(df):,} -> {len(bal):,} (kehilangan {(1-len(bal)/len(df))*100:.1f}% demi kesetaraan topik & panjang)")

sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=SEED)
fold = np.full(len(bal), -1)
for k, (_, te_idx) in enumerate(sgkf.split(bal, bal.label, bal.group_id)):
    fold[te_idx] = k
edges = np.cumsum([CFG["TRAIN_FOLDS"], CFG["VAL_FOLDS"], CFG["DEVTEST_FOLDS"]])
bal["split"] = np.select([fold < edges[0], fold < edges[1], fold < edges[2]], ["train", "val", "dev_test"], "lockbox")
bal["id"] = [f"POL2-{i+1:06d}" for i in range(len(bal))]
print(pd.crosstab(bal.split, bal.label).assign(total=lambda x: x.sum(axis=1)).to_string())
print("\nLOCKBOX disegel: hanya jumlahnya yang dilihat. Isi/statistiknya TIDAK dilaporkan sampai protokol selesai.")

[6_seimbang_topik_panjang    ] total=  3,494 | per label={0: 1747, 1: 1747}  strata = sub_topic x len_bin
Ukuran: 25,772 -> 3,494 (kehilangan 86.4% demi kesetaraan topik & panjang)
label        0     1  total
split                      
dev_test   174   174    348
lockbox    175   174    349
train     1223  1225   2448
val        175   174    349

LOCKBOX disegel: hanya jumlahnya yang dilihat. Isi/statistiknya TIDAK dilaporkan sampai protokol selesai.


In [10]:
# ---------------- 7. VERIFIKASI TERHADAP KRITERIA YANG DITETAPKAN DI AWAL ----
res = []
dev = bal[bal.split != "lockbox"]                      # lockbox tidak ikut diaudit
tr_, va_, te_ = (bal[bal.split == s] for s in ("train", "val", "dev_test"))

for s, d in (("train", tr_), ("val", va_), ("dev_test", te_)):
    r = d.groupby("label").text.apply(lambda x: x.map(tp.dup_title).mean() * 100)
    res.append((f"C1 judul ganda [{s}]", abs(r[0] - r[1]), 5, abs(r[0] - r[1]) <= 5))
ks = ks_2samp(dev[dev.label == 0].n_words, dev[dev.label == 1].n_words).statistic
res.append(("C2 KS panjang kata", ks, 0.10, ks <= 0.10))
e = dev.groupby("label").text.apply(lambda x: x.str.endswith((".", "!", "?")).mean() * 100)
res.append(("C3 berakhir tanda baca", abs(e[0] - e[1]), 5, abs(e[0] - e[1]) <= 5))

def lenfeat(d):
    t = d.text; w = t.str.split().str.len()
    return np.c_[w, t.str.len(), t.str.count(r"[.!?]"), t.str.count(",")]
dt = DecisionTreeClassifier(max_depth=4, random_state=0).fit(lenfeat(tr_), tr_.label)
acc_len = accuracy_score(te_.label, dt.predict(lenfeat(te_)))
res.append(("C4 akurasi fitur panjang (dev_test)", acc_len, 0.60, acc_len <= 0.60))

viol = bal.groupby("group_id").split.nunique().gt(1).sum()
res.append(("C5 grup melintasi split", int(viol), 0, viol == 0))
tp_ = dev.groupby("label").sub_topic.value_counts(normalize=True).unstack(0).fillna(0)
dtopic = (tp_[0] - tp_[1]).abs().max() * 100
res.append(("C6 selisih proporsi sub-topik", dtopic, 2, dtopic <= 2))

print(f"{'Kriteria':<36}{'Nilai':>10}{'Ambang':>9}  Hasil")
for name, v, th, ok in res:
    print(f"{name:<36}{v:>10.3f}{th:>9}  {'LULUS' if ok else 'GAGAL'}")

first8 = lambda d: d.text.str.split().str[:8].str.join(" ")
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
lr = LogisticRegression(max_iter=2000).fit(vec.fit_transform(first8(tr_)), tr_.label)
acc8 = accuracy_score(te_.label, lr.predict(vec.transform(first8(te_))))
print(f"\nINFO akurasi 8 kata pertama (TF-IDF+LR, dev_test): {acc8*100:.2f}%  <- bila tinggi, gaya penulisan/sumber masih membedakan kelas")
print("INFO sumber x label (ketimpangan sumber TIDAK dapat diperbaiki oleh kode; perlu sumber tambahan):")
print(pd.crosstab(dev.source, dev.label).to_string())   # tanpa lockbox

have = [l for l in (0, 1) if dev[dev.label == l].date.notna().any()]
if len(have) == 2:
    yr = pd.crosstab(dev.date.dt.year, dev.label); yr = yr.div(yr.sum(axis=0), axis=1)
    overlap = float(np.minimum(yr[0], yr[1]).sum()) * 100
    print(f"INFO tumpang-tindih distribusi tahun (kolom tanggal): {overlap:.1f}%  (100% = identik; 0% = terpisah total)")
    if overlap < 50: print("   >> PERINGATAN: periode kedua kelas hampir tidak beririsan; model dapat memakai 'zaman' sebagai petunjuk label.")
else:
    print(f"INFO tahun dari kolom tanggal: TIDAK DAPAT DIHITUNG; kelas tanpa tanggal terbaca: {[l for l in (0, 1) if l not in have]}."
          " (Ini keterbatasan data, BUKAN bukti periode terpisah. Periksa nama kolom tanggal di berkas mentah.)")
yrs = dev.text.str.findall(r"\b(20[12]\d)\b")
ym = pd.DataFrame({l: pd.Series([y for ys in yrs[dev.label == l] for y in set(ys)]).value_counts() / (dev.label == l).sum() * 100 for l in (0, 1)}).fillna(0).round(1).sort_index()
print("\nINFO tahun yang DISEBUT dalam teks (% teks), proksi zaman:"); print(ym.to_string())
show_retention()
audit(dev, "AUDIT SETELAH PENYEIMBANGAN (tanpa lockbox)")
ALL_OK = all(ok for *_, ok in res)
if not ALL_OK:
    print("\n" + "=" * 78); print("DIAGNOSIS OTOMATIS KRITERIA GAGAL (hanya membaca data train/val/dev_test, bukan lockbox)"); print("=" * 78)
    if any(n_.startswith("C1") and not ok for n_, _, _, ok in res):
        for lab in (0, 1):
            sd = dev[(dev.label == lab) & dev.text.map(tp.dup_title)]
            print(f"\nlabel {lab}: {len(sd):,} teks berjudul ganda | per sumber: {sd.source.value_counts().to_dict()}")
            for t_ in sd.text.head(4): print("   -", t_[:170])
    if any(n_.startswith("C4") and not ok for n_, _, _, ok in res):
        names = ["n_kata", "n_karakter", "n_titik/tanya/seru", "n_koma"]
        Xtr_, Xte_ = lenfeat(tr_), lenfeat(te_)
        print("\nC4: akurasi pohon (kedalaman 3) per SATU fitur, dan rata-rata fitur per label:")
        for i, nm in enumerate(names):
            a_ = accuracy_score(te_.label, DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr_[:, [i]], tr_.label).predict(Xte_[:, [i]]))
            print(f"   {nm:<20} akurasi={a_*100:5.1f}% | rata2 fakta={Xtr_[tr_.label.values==0, i].mean():7.2f} | rata2 hoaks={Xtr_[tr_.label.values==1, i].mean():7.2f}")
        print("   Bila satu fitur (mis. n_koma) dominan: itu gaya penulisan sumber, bukan panjang. Putuskan di PROTOKOL: cocokkan strata tambahan ATAU laporkan sebagai keterbatasan.")
print("\nSEMUA KRITERIA LULUS" if ALL_OK else "\nADA KRITERIA GAGAL -> perbaiki pipeline, JANGAN longgarkan ambang tanpa alasan tertulis.")

Kriteria                                 Nilai   Ambang  Hasil
C1 judul ganda [train]                  12.974        5  GAGAL
C1 judul ganda [val]                    12.085        5  GAGAL
C1 judul ganda [dev_test]                6.322        5  GAGAL
C2 KS panjang kata                       0.083      0.1  LULUS
C3 berakhir tanda baca                   0.000        5  LULUS
C4 akurasi fitur panjang (dev_test)      0.759      0.6  GAGAL
C5 grup melintasi split                  0.000        0  LULUS
C6 selisih proporsi sub-topik            0.735        2  LULUS

INFO akurasi 8 kata pertama (TF-IDF+LR, dev_test): 85.34%  <- bila tinggi, gaya penulisan/sumber masih membedakan kelas
INFO sumber x label (ketimpangan sumber TIDAK dapat diperbaiki oleh kode; perlu sumber tambahan):
label               0     1
source                     
cnnindonesia.com  557     0
hoax_archive_15k    0   264
komdigi.go.id       0     1
kompas.com        413     0
tempo.co          602     0
turnbackhoax.id   

In [11]:
# ---------------- 8. SIMPAN (berkas baru; tidak menimpa data lama) + MANIFEST -
bal = bal.drop(columns=["title", "body"], errors="ignore").rename(columns={"title_c": "title", "body_c": "body"})
cols = ["id", "text", "title", "body", "label", "sub_topic", "source", "group_id", "date"]
for sp in ("train", "val", "dev_test"):
    bal[bal.split == sp][cols].to_csv(os.path.join(OUT_DIR, f"{sp}.csv"), index=False, encoding="utf-8-sig")
bal[bal.split != "lockbox"][cols + ["split"]].to_csv(os.path.join(OUT_DIR, "dataset_v2_full.csv"), index=False, encoding="utf-8-sig")
LOCK_DIR = os.path.join(OUT_DIR, "LOCKBOX"); os.makedirs(LOCK_DIR, exist_ok=True)
bal[bal.split == "lockbox"][cols].to_csv(os.path.join(LOCK_DIR, "lockbox.csv"), index=False, encoding="utf-8-sig")
open(os.path.join(LOCK_DIR, "JANGAN_DIBUKA.txt"), "w", encoding="utf-8").write(
    "Set uji tersegel. Dibuka SATU KALI untuk pelaporan akhir, setelah model, ambang, dan analisis dikunci di PROTOKOL.\n"
    "Tidak boleh dipakai untuk tuning, pemilihan model, atau diagnosis. Catat pembukaan di PROTOKOL (tanggal + hash).\n")
pd.DataFrame(LOG).to_csv(os.path.join(OUT_DIR, "audit_log_per_langkah.csv"), index=False)
pd.DataFrame([(n_, v, th, ok) for n_, v, th, ok in res], columns=["kriteria", "nilai", "ambang", "lulus"]).to_csv(
    os.path.join(OUT_DIR, "hasil_kriteria.csv"), index=False)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for ch in iter(lambda: f.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()

cfg_out = dict(CFG, pipeline_version=tp.PIPELINE_VERSION, python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
               created=datetime.datetime.now().isoformat(timespec="seconds"), raw_dir=RAW_DIR, all_criteria_passed=bool(ALL_OK))
json.dump(cfg_out, open(os.path.join(OUT_DIR, "config.json"), "w"), indent=2)

man = [{"berkas": f, "sha256": sha(os.path.join(OUT_DIR, f))} for f in sorted(os.listdir(OUT_DIR)) if f not in ("MANIFEST_sha256.csv", "LOCKBOX")]
man += [{"berkas": "RAW/" + f, "sha256": sha(os.path.join(RAW_DIR, f))} for f in sorted(df_f for df_f in raw["file"].unique())]
man += [{"berkas": "LOCKBOX/lockbox.csv", "sha256": sha(os.path.join(LOCK_DIR, "lockbox.csv"))}]
man += [{"berkas": "text_pipeline.py", "sha256": sha("text_pipeline.py")}]
pd.DataFrame(man).to_csv(os.path.join(OUT_DIR, "MANIFEST_sha256.csv"), index=False)
print("Tersimpan di:", os.path.abspath(OUT_DIR)); print(sorted(os.listdir(OUT_DIR)))

Tersimpan di: e:\SEMESTER 5\TA\PROJECT\data\processed_v2
['LOCKBOX', 'MANIFEST_sha256.csv', 'audit_log_per_langkah.csv', 'config.json', 'dataset_v2_full.csv', 'dev_test.csv', 'hasil_kriteria.csv', 'train.csv', 'val.csv']
